In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import (
    accuracy_score, classification_report,
    confusion_matrix, ConfusionMatrixDisplay,
    roc_auc_score, roc_curve
)
from sklearn.decomposition import PCA

plt.rcParams['figure.dpi'] = 120
print('Libraries loaded ✅')

## 1. Load the Dataset

In [2]:
df = pd.read_csv('data.csv')

df = df.drop(columns=['id', 'Unnamed: 32'])

print(f'Shape: {df.shape}')
print(f'\nMissing values: {df.isnull().sum().sum()}')
print(f'\nDiagnosis distribution:')
print(df['diagnosis'].value_counts())
print(f'\nMalignant rate: {(df["diagnosis"]=="M").mean()*100:.1f}%')
df.head()

## 2. Preprocessing

In [3]:
X = df.drop(columns=['diagnosis'])
y = (df['diagnosis'] == 'M').astype(int)

feature_names = X.columns.tolist()
print(f'Features : {len(feature_names)}')
print(f'Samples  : {len(X)}')

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler   = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print(f'\nTrain: {X_train_s.shape} | Test: {X_test_s.shape}')
print(f'Malignant in test: {y_test.sum()} ({y_test.mean()*100:.1f}%)')

## 3. SVM — Kernel Comparison

In [4]:
kernels = ['linear', 'rbf', 'poly']
kernel_results = {}

for kernel in kernels:
    svm = SVC(kernel=kernel, probability=True, random_state=42)
    svm.fit(X_train_s, y_train)
    y_pred = svm.predict(X_test_s)
    y_prob = svm.predict_proba(X_test_s)[:, 1]

    kernel_results[kernel] = {
        'model'  : svm,
        'y_pred' : y_pred,
        'y_prob' : y_prob,
        'acc'    : accuracy_score(y_test, y_pred),
        'auc'    : roc_auc_score(y_test, y_prob),
    }
    print(f'SVM ({kernel:<6}) — Accuracy: {kernel_results[kernel]["acc"]:.4f}  AUC: {kernel_results[kernel]["auc"]:.4f}')

In [5]:
for kernel in kernels:
    print(f'\n=== SVM — {kernel.upper()} kernel ===')
    print(classification_report(
        y_test, kernel_results[kernel]['y_pred'],
        target_names=['Benign', 'Malignant']
    ))

## 4. Confusion Matrices

In [6]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
fig.suptitle('SVM — Confusion Matrices by Kernel', fontsize=13, fontweight='bold')

cmaps = {'linear': 'Blues', 'rbf': 'Greens', 'poly': 'Oranges'}

for ax, kernel in zip(axes, kernels):
    cm   = confusion_matrix(y_test, kernel_results[kernel]['y_pred'])
    disp = ConfusionMatrixDisplay(cm, display_labels=['Benign', 'Malignant'])
    disp.plot(ax=ax, colorbar=False, cmap=cmaps[kernel])
    acc = kernel_results[kernel]['acc']
    ax.set_title(f'{kernel.upper()} kernel\nAcc={acc:.4f}', fontweight='bold')

plt.tight_layout()
plt.show()

## 5. ROC Curves & AUC

In [7]:
fig, ax = plt.subplots(figsize=(8, 6))

colors = {'linear': '#2196F3', 'rbf': '#4CAF50', 'poly': '#FF9800'}

for kernel in kernels:
    fpr, tpr, _ = roc_curve(y_test, kernel_results[kernel]['y_prob'])
    auc = kernel_results[kernel]['auc']
    ax.plot(fpr, tpr, label=f'{kernel.upper()} (AUC={auc:.3f})',
            color=colors[kernel], linewidth=2)

ax.plot([0,1],[0,1],'k:', linewidth=1, label='Random')
ax.set_xlabel('False Positive Rate')
ax.set_ylabel('True Positive Rate')
ax.set_title('ROC Curves — SVM Kernels', fontweight='bold')
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Hyperparameter Tuning — GridSearchCV

In [8]:
param_grid = {
    'C'    : [0.1, 1, 10, 100],
    'gamma': ['scale', 'auto', 0.001, 0.01, 0.1],
}

grid_search = GridSearchCV(
    SVC(kernel='rbf', probability=True, random_state=42),
    param_grid,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)
grid_search.fit(X_train_s, y_train)

print(f'Best params   : {grid_search.best_params_}')
print(f'Best CV acc   : {grid_search.best_score_:.4f}')

best_svm = grid_search.best_estimator_
y_pred_best = best_svm.predict(X_test_s)
y_prob_best = best_svm.predict_proba(X_test_s)[:, 1]

print(f'\nTuned SVM test accuracy : {accuracy_score(y_test, y_pred_best):.4f}')
print(f'Tuned SVM AUC           : {roc_auc_score(y_test, y_prob_best):.4f}')
print()
print(classification_report(y_test, y_pred_best, target_names=['Benign','Malignant']))

## 7. Cross-Validation (5-Fold)

In [9]:
cv_results = {}
for kernel in kernels:
    svm = SVC(kernel=kernel, probability=True, random_state=42)
    scores = cross_val_score(svm, X_train_s, y_train, cv=5, scoring='accuracy')
    cv_results[kernel] = scores
    print(f'SVM {kernel:<6} CV: {scores.mean():.4f} ± {scores.std():.4f}  | scores: {scores.round(4)}')

scores_tuned = cross_val_score(best_svm, X_train_s, y_train, cv=5, scoring='accuracy')
cv_results['rbf_tuned'] = scores_tuned
print(f'SVM rbf (tuned) CV: {scores_tuned.mean():.4f} ± {scores_tuned.std():.4f}  | scores: {scores_tuned.round(4)}')

## 8. Feature Importance — Linear SVM

In [10]:
linear_svm = kernel_results['linear']['model']
coef = pd.Series(np.abs(linear_svm.coef_[0]), index=feature_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(10, 6))
coef.head(15)[::-1].plot(kind='barh', ax=ax, color='steelblue')
ax.set_title('Top 15 Most Important Features — Linear SVM', fontweight='bold')
ax.set_xlabel('|Coefficient|')
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('Top 10 features:')
print(coef.head(10).to_string())

## 9. Decision Boundary Visualisation (PCA 2D)

In [11]:
pca = PCA(n_components=2, random_state=42)
X_train_pca = pca.fit_transform(X_train_s)
X_test_pca  = pca.transform(X_test_s)

svm_pca = SVC(kernel='rbf', C=grid_search.best_params_['C'],
              gamma=grid_search.best_params_['gamma'], random_state=42)
svm_pca.fit(X_train_pca, y_train)

h = 0.1
x_min, x_max = X_train_pca[:, 0].min() - 1, X_train_pca[:, 0].max() + 1
y_min, y_max = X_train_pca[:, 1].min() - 1, X_train_pca[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, h),
                     np.arange(y_min, y_max, h))
Z = svm_pca.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)

fig, ax = plt.subplots(figsize=(9, 6))
ax.contourf(xx, yy, Z, alpha=0.25, cmap='coolwarm')
ax.contour(xx, yy, Z, colors='black', linewidths=0.8, linestyles='--')

colors_map = {0: '#2196F3', 1: '#E53935'}
labels_map = {0: 'Benign', 1: 'Malignant'}
for cls in [0, 1]:
    mask = y_train == cls
    ax.scatter(X_train_pca[mask, 0], X_train_pca[mask, 1],
               c=colors_map[cls], label=labels_map[cls], alpha=0.5, s=20)

ax.set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% variance)')
ax.set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% variance)')
ax.set_title('SVM RBF — Decision Boundary (PCA 2D)', fontweight='bold')
ax.legend()
ax.grid(alpha=0.2)
plt.tight_layout()
plt.show()

## 10. Summary / Comparison

In [12]:
rows = []
for kernel in kernels:
    rows.append({
        'Model'   : f'SVM — {kernel}',
        'Accuracy': kernel_results[kernel]['acc'],
        'AUC'     : kernel_results[kernel]['auc'],
        'CV Mean' : cv_results[kernel].mean(),
        'CV Std'  : cv_results[kernel].std(),
    })
rows.append({
    'Model'   : 'SVM — rbf (tuned)',
    'Accuracy': accuracy_score(y_test, y_pred_best),
    'AUC'     : roc_auc_score(y_test, y_prob_best),
    'CV Mean' : cv_results['rbf_tuned'].mean(),
    'CV Std'  : cv_results['rbf_tuned'].std(),
})

summary = pd.DataFrame(rows).round(4)
print('=' * 65)
print('  SVM MODEL COMPARISON SUMMARY')
print('=' * 65)
print(summary.to_string(index=False))
print('=' * 65)

In [13]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models = summary['Model'].tolist()
x = np.arange(len(models))
width = 0.35

axes[0].bar(x, summary['Accuracy'], color=['#2196F3','#4CAF50','#FF9800','#9C27B0'],
            edgecolor='white', width=0.55)
for i, v in enumerate(summary['Accuracy']):
    axes[0].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=8, rotation=10)
axes[0].set_ylim(0.85, 1.02)
axes[0].set_title('Test Accuracy', fontweight='bold')
axes[0].set_ylabel('Accuracy')
axes[0].grid(axis='y', alpha=0.3)

axes[1].bar(x, summary['AUC'], color=['#2196F3','#4CAF50','#FF9800','#9C27B0'],
            edgecolor='white', width=0.55)
for i, v in enumerate(summary['AUC']):
    axes[1].text(i, v + 0.002, f'{v:.4f}', ha='center', fontsize=9, fontweight='bold')
axes[1].set_xticks(x)
axes[1].set_xticklabels(models, fontsize=8, rotation=10)
axes[1].set_ylim(0.85, 1.02)
axes[1].set_title('AUC-ROC', fontweight='bold')
axes[1].set_ylabel('AUC')
axes[1].grid(axis='y', alpha=0.3)

plt.suptitle('SVM — Accuracy & AUC by Model Configuration', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()